In [9]:
"""
=============================================================================
  F-18 HORNET — DDPG v3 — FIXED & ROBUST

  ROOT CAUSE FIXES vs v2:
  ─────────────────────────────────────────────────────────────────────
  FIX-1  ACTION SPACE REDESIGN
         Old: (dx,dy,dz) in metres → autopilot tries to fly there in 1 step
         Problem: dz=±30m every 0.5s = 60 m/s climb rate → instant alt violation
         New: action = (psi_rate_cmd, bank_rate_cmd, dz_trim)
              psi_rate_cmd  ∈ [-1,+1] → [-PSI_DOT_MAX, +PSI_DOT_MAX] deg/s
              bank_rate_cmd ∈ [-1,+1] → [-BANK_RATE,   +BANK_RATE]   deg/s
              dz_trim       ∈ [-1,+1] → [-DZ_MAX,       +DZ_MAX]      m/s vertical
         This gives the agent DIRECT control authority over heading rate
         and makes altitude deviations physically bounded.

  FIX-2  ALTITUDE CONTROLLER REWRITE
         Previous dynamics used dz_cmd as a position delta → chaos
         New: dz_trim feeds directly into a proportional altitude hold
         Altitude hold autopilot: hdot_cmd = Kh*(z_target - z) + dz_trim
         This means even random actions keep the aircraft near 5000m.

  FIX-3  REWARD NORMALISATION
         All reward components scaled to [-1, +1] per step before weighting.
         No component can dominate by raw magnitude.
         Total step reward nominally in [-50, +100] range.

  FIX-4  TIGHTER TERMINATION
         alt_strikes=3 (not 5 or 10) with immediate −600 terminal.
         crash floor raised to 3000m so we catch it before physics blows up.

  FIX-5  HARD CONSTRAINTS ENCODED AS SHAPED BARRIERS
         Altitude:    tanh-barrier, peaks sharply at ±50m boundary
         Load factor: reward band 3–7g with cliff outside
         Speed:       reward band [230,290] m/s

  FIX-6  STATE NORMALISATION
         Observation fed to networks is normalised to ~[-1,+1]:
         positions /= 5000, angles /= 180, velocities /= 300, rates /= 30

  FIX-7  CURRICULUM
         Episodes 0-1000:  only heading+altitude objectives (learn to fly)
         Episodes 1001+:   full reward including position, aggression, time
=============================================================================
"""

import math, random, copy, time, os, warnings
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

warnings.filterwarnings("ignore")

try:
    import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[DDPG-v3]  PyTorch {torch.__version__}  device={DEVICE}")
except ImportError:
    raise SystemExit("pip install torch")

# ══════════════════════════════════════════════════════════════════════════════
# CONSTANTS
# ══════════════════════════════════════════════════════════════════════════════
G            = 9.81
V_CRUISE     = 260.0       # m/s
Z_CRUISE     = 5000.0      # m
Z_BAND       = 50.0        # m  hard altitude wall  ← ENFORCED
NZ_MIN       = 3.0         # g  floor
NZ_MAX       = 7.0         # g  structural ceiling
NZ_SWEET_LO  = 4.5         # g  reward band low
NZ_SWEET_HI  = 6.8         # g  reward band high
PHI_LIMIT    = 80.0        # deg
PHI_AGGR     = 65.0        # deg  aggression threshold

PHI_OPT_DEG  = min(PHI_LIMIT, math.degrees(math.acos(1.0/NZ_MAX)))
PSI_DOT_MAX  = math.degrees(G * math.tan(math.radians(PHI_OPT_DEG)) / V_CRUISE)
R_OPT        = V_CRUISE / math.radians(PSI_DOT_MAX)

STATE_DIM    = 12
ACTION_DIM   = 3
DT           = 0.5         # s

# FIX-1: Direct heading-rate action space
A_PSI_DOT    = PSI_DOT_MAX          # deg/s  max commanded turn rate
A_BANK_RATE  = 60.0                 # deg/s  max bank rate command
A_DZ_TRIM    = 3.0                  # m/s    altitude trim authority (tight!)

HEADINGS     = [0, 30, 60, 90, 120, 150, 180]

# Altitude controller gains
KH_HOLD      = 0.8    # proportional gain for altitude hold
KH_MAX_HDOT  = 5.0    # m/s  max vertical speed commanded by autopilot

print(f"[Physics] PHI_OPT={PHI_OPT_DEG:.1f}°  PSI_DOT_MAX={PSI_DOT_MAX:.1f}°/s  "
      f"R_OPT={R_OPT:.0f}m  NZ_design={1/math.cos(math.radians(PHI_OPT_DEG)):.2f}g")
print(f"[Actions] psi_dot=±{A_PSI_DOT:.1f}°/s  bank_rate=±{A_BANK_RATE:.0f}°/s  "
      f"dz_trim=±{A_DZ_TRIM:.1f}m/s")

# ══════════════════════════════════════════════════════════════════════════════
# MANEUVER SPEC
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ManeuverSpec:
    psi0: float; psi1: float; delta: float
    key: str; label: str; category: str
    phi_opt: float; psi_dot_opt: float; R: float
    t_opt: float; t_limit: float; nz_design: float
    x_target: float; y_target: float; z_target: float = Z_CRUISE
    success_herr: float = 5.0
    success_hold: int   = 4
    alt_strikes:  int   = 3          # FIX-4: fast termination
    extra: dict = field(default_factory=dict)


def _arc_endpoint(psi0_deg, delta_deg, R):
    psi0_r = math.radians(psi0_deg); d_r = math.radians(delta_deg)
    x_arc  = R*(1-math.cos(d_r)); y_arc = R*math.sin(d_r)
    s0=math.sin(psi0_r); c0=math.cos(psi0_r)
    return x_arc*c0+y_arc*s0,  -x_arc*s0+y_arc*c0


def build_specs():
    specs = {}
    for i,psi0 in enumerate(HEADINGS):
        for psi1 in HEADINGS[i+1:]:
            delta  = float(psi1-psi0)
            phi_r  = math.radians(PHI_OPT_DEG)
            pd     = math.degrees(G*math.tan(phi_r)/V_CRUISE)
            R      = V_CRUISE/math.radians(pd)
            t_opt  = delta/pd
            xt,yt  = _arc_endpoint(psi0,delta,R)
            dist   = math.hypot(xt,yt)
            cat    = 'nav' if delta<=60 else ('aggressive' if delta<=120 else 'extreme')
            key    = f"H{psi0:03d}_{psi1:03d}"
            specs[key] = ManeuverSpec(
                psi0=float(psi0), psi1=float(psi1), delta=delta,
                key=key, label=f"{psi0}°→{psi1}° ({delta:.0f}°)",
                category=cat, phi_opt=PHI_OPT_DEG, psi_dot_opt=pd,
                R=R, t_opt=t_opt, t_limit=max(2.5*t_opt, 15.0),
                nz_design=1/math.cos(phi_r),
                x_target=xt, y_target=yt,
                success_herr=max(3.0, delta*0.07),
                extra=dict(dist_init=dist)
            )
    print(f"[Specs] {len(specs)} maneuvers built")
    return specs

ALL_SPECS = build_specs()

# ══════════════════════════════════════════════════════════════════════════════
# F-18 6-DOF DYNAMICS  — rewritten altitude controller (FIX-2)
# ══════════════════════════════════════════════════════════════════════════════

class F18_6DOF:
    MASS=16651; Ixx=23000; Iyy=169000; Izz=185000; Ixz=1500
    WING_AREA=37.16; SPAN=11.43; CHORD=3.51
    RHO_SL=1.225; H_SCALE=8500
    CL0=0.10; CL_a=5.0; CL_q=4.0
    CD0=0.022; K_ind=0.16
    CY_b=-0.90; CY_r=0.30
    Cl_p=-0.45; Cl_r=0.18; Cl_b=-0.08
    Cm_a=-1.20; Cm_q=-8.0; Cm0=0.02
    Cn_b=0.18; Cn_p=-0.08; Cn_r=-0.25
    V_MIN=150.0; V_MAX=450.0; NZ_CAP=9.0
    P_MAX=100.0; Q_MAX=25.0; R_MAX=20.0
    THETA_MAX=25.0; THETA_MIN=-15.0

    def __init__(self, dt=DT): self.dt=dt; self.reset()

    def reset(self, psi0=0.0):
        self.x=0.; self.y=0.; self.z=Z_CRUISE
        self.psi=float(psi0); self.theta=0.; self.phi=0.
        self.u=V_CRUISE; self.v=0.; self.w=0.
        self.p=0.; self.q=0.; self.r=0.
        self.V=V_CRUISE; self.alpha=0.; self.beta=0.; self.nz=1.
        self.steps=0; self.elapsed=0.

    def _rho(self): return self.RHO_SL*math.exp(-max(self.z,0)/self.H_SCALE)

    def _aero(self):
        rho=self._rho(); V=max(self.V,1.)
        self.alpha=math.degrees(math.atan2(self.w,self.u))
        self.beta =math.degrees(math.asin(np.clip(self.v/V,-1.,1.)))
        ar=math.radians(self.alpha); br=math.radians(self.beta)
        qb=0.5*rho*V**2; S=self.WING_AREA; b=self.SPAN; c=self.CHORD
        pn=math.radians(self.p)*b/(2*V)
        qn=math.radians(self.q)*c/(2*V)
        rn=math.radians(self.r)*b/(2*V)
        CL=min(self.CL0+self.CL_a*ar+self.CL_q*qn, 1.8)
        CD=self.CD0+self.K_ind*CL**2
        CY=self.CY_b*br+self.CY_r*rn
        Cl_=self.Cl_p*pn+self.Cl_r*rn+self.Cl_b*br
        Cm_=self.Cm0+self.Cm_a*ar+self.Cm_q*qn
        Cn_=self.Cn_b*br+self.Cn_p*pn+self.Cn_r*rn
        ca=math.cos(ar); sa=math.sin(ar); cb=math.cos(br)
        L=qb*S*CL; D=qb*S*CD; Y=qb*S*CY
        Xa=-D*ca*cb+L*sa; Ya=Y; Za=-D*sa*cb-L*ca
        Lm=qb*S*b*Cl_; Ma=qb*S*c*Cm_; Na=qb*S*b*Cn_
        return Xa,Ya,Za,Lm,Ma,Na,L

    def step(self, psi_dot_cmd_deg, bank_rate_cmd_deg, hdot_cmd_ms):
        """
        FIX-2: Direct angular rate commands instead of position deltas.
        psi_dot_cmd_deg : desired heading rate [deg/s]
        bank_rate_cmd_deg: desired bank rate   [deg/s]
        hdot_cmd_ms     : desired vertical speed [m/s]  (bounded ±DZ_MAX)
        """
        self.steps += 1; self.elapsed += self.dt
        dt = self.dt

        # ── Bank angle autopilot ──────────────────────────────────────
        phi_r     = math.radians(self.phi)
        theta_r   = math.radians(self.theta)
        # Desired bank from commanded psi_dot: phi_des = atan(psi_dot*V/g)
        phi_des_from_turn = math.degrees(
            math.atan(math.radians(psi_dot_cmd_deg)*self.V/G))
        phi_des   = np.clip(phi_des_from_turn + bank_rate_cmd_deg*dt,
                            -PHI_LIMIT, PHI_LIMIT)
        dphi      = np.clip(phi_des - self.phi, -60.*dt, 60.*dt)
        self.phi  = np.clip(self.phi + dphi, -PHI_LIMIT, PHI_LIMIT)

        # ── Altitude hold autopilot (FIX-2) ──────────────────────────
        # Proportional controller + agent trim input
        z_err_now  = Z_CRUISE - self.z    # positive when below cruise
        hdot_auto  = KH_HOLD * z_err_now  # proportional term
        hdot_total = np.clip(hdot_auto + hdot_cmd_ms, -KH_MAX_HDOT, KH_MAX_HDOT)
        # Convert to pitch demand
        theta_des  = np.clip(math.degrees(math.asin(
            np.clip(hdot_total/max(self.V,1.), -0.3, 0.3))),
            self.THETA_MIN, self.THETA_MAX)
        dtheta     = np.clip(theta_des - self.theta, -15.*dt, 15.*dt)
        self.theta = np.clip(self.theta + dtheta, self.THETA_MIN, self.THETA_MAX)

        # ── Aero forces ───────────────────────────────────────────────
        Xa,Ya,Za,Lm,Ma,Na,Lift = self._aero()
        phi_r   = math.radians(self.phi); theta_r = math.radians(self.theta)
        Gx=-G*math.sin(theta_r)
        Gy= G*math.cos(theta_r)*math.sin(phi_r)
        Gz= G*math.cos(theta_r)*math.cos(phi_r)

        # Thrust to hold speed
        T = np.clip(self.MASS*(G*math.sin(theta_r) + 0.3*(V_CRUISE-self.V)),
                    0., self.MASS*50.)
        pr=math.radians(self.p); qr=math.radians(self.q); rr=math.radians(self.r)
        ax=(Xa+T)/self.MASS+Gx-(qr*self.w-rr*self.v)
        ay=Ya/self.MASS+Gy-(rr*self.u-pr*self.w)
        az=Za/self.MASS+Gz-(pr*self.v-qr*self.u)
        self.u=np.clip(self.u+ax*dt,-self.V_MAX,self.V_MAX)
        self.v=np.clip(self.v+ay*dt,-50.,50.)
        self.w=np.clip(self.w+az*dt,-60.,60.)

        # ── Body rates ────────────────────────────────────────────────
        Ix=self.Ixx; Iy=self.Iyy; Iz=self.Izz; Ixz_=self.Ixz
        Gam=Ix*Iz-Ixz_**2
        pd=(Iz*Lm+Ixz_*Na-(Iz*(Iz-Iy)*rr+Ixz_*(Ix-Iy+Iz)*pr)*qr)/Gam
        qd=(Ma-(Ix-Iz)*pr*rr-Ixz_*(pr**2-rr**2))/Iy
        rd=(Ix*Na+Ixz_*Lm+(Ix*(Ix-Iy)*pr+Ixz_*(Ix-Iy+Iz)*rr)*qr)/Gam
        self.p=np.clip(self.p+math.degrees(pd)*dt,-self.P_MAX,self.P_MAX)
        self.q=np.clip(self.q+math.degrees(qd)*dt,-self.Q_MAX,self.Q_MAX)
        self.r=np.clip(self.r+math.degrees(rd)*dt,-self.R_MAX,self.R_MAX)

        # ── Euler kinematics ──────────────────────────────────────────
        phi_r=math.radians(self.phi); theta_r=math.radians(self.theta)
        pr=math.radians(self.p); qr=math.radians(self.q); rr=math.radians(self.r)
        ct=math.cos(theta_r); cf=math.cos(phi_r); sf=math.sin(phi_r)
        psi_dot2=(qr*sf+rr*cf)/(ct+1e-9)
        theta_d2=qr*cf-rr*sf
        phi_d2  =pr+(qr*sf+rr*cf)*math.tan(theta_r)
        self.psi  =(self.psi+math.degrees(psi_dot2)*dt)%360.
        self.theta=np.clip(self.theta+math.degrees(theta_d2)*dt,self.THETA_MIN,self.THETA_MAX)
        self.phi  =np.clip(self.phi+math.degrees(phi_d2)*dt,-PHI_LIMIT,PHI_LIMIT)

        # ── Position update ───────────────────────────────────────────
        phi_r=math.radians(self.phi); theta_r=math.radians(self.theta)
        psi_r=math.radians(self.psi)
        cp=math.cos(psi_r); sp=math.sin(psi_r)
        ct2=math.cos(theta_r); st2=math.sin(theta_r)
        cf2=math.cos(phi_r); sf2=math.sin(phi_r)
        vn=(ct2*cp)*self.u+(sf2*st2*cp-cf2*sp)*self.v+(cf2*st2*cp+sf2*sp)*self.w
        ve=(ct2*sp)*self.u+(sf2*st2*sp+cf2*cp)*self.v+(cf2*st2*sp-sf2*cp)*self.w
        vd=(-st2)*self.u+(sf2*ct2)*self.v+(cf2*ct2)*self.w
        self.x+=ve*dt; self.y+=vn*dt; self.z-=vd*dt

        # ── Derived ───────────────────────────────────────────────────
        self.nz=np.clip(Lift*math.cos(phi_r)/(self.MASS*G), 0.1, self.NZ_CAP)
        self.V =math.sqrt(self.u**2+self.v**2+self.w**2)
        return self.sv()

    def sv(self):
        return dict(x=self.x,y=self.y,z=self.z,
                    psi=self.psi,theta=self.theta,phi=self.phi,
                    u=self.u,v=self.v,w=self.w,p=self.p,q=self.q,r=self.r,
                    V=self.V,alpha=self.alpha,beta=self.beta,nz=self.nz,
                    elapsed=self.elapsed)

    def state_vec(self):
        # FIX-6: normalised observation
        return np.array([
            self.x/5000., self.y/5000., (self.z-Z_CRUISE)/Z_BAND,
            self.psi/180., self.theta/30., self.phi/PHI_LIMIT,
            (self.u-V_CRUISE)/100., self.v/50., self.w/30.,
            self.p/self.P_MAX, self.q/self.Q_MAX, self.r/self.R_MAX
        ], dtype=np.float32)


# ══════════════════════════════════════════════════════════════════════════════
# HEADING ERROR
# ══════════════════════════════════════════════════════════════════════════════
def herr(psi, target):
    e = target - psi
    while e >  180.: e -= 360.
    while e <= -180.: e += 360.
    return e   # signed


# ══════════════════════════════════════════════════════════════════════════════
# REWARD  v3  — normalised, barrier-based, curriculum-aware
# ══════════════════════════════════════════════════════════════════════════════

def compute_reward(sv_prev, sv, spec, action, done, cause, curriculum_phase):
    """
    All per-step components normalised to [-1, +1] before weighting.

    PHASE 0 (ep < 1000): Only altitude + heading active → learn to fly straight/level
    PHASE 1 (ep ≥ 1000): Full reward — position, aggression, time urgency

    Weights (tuned so no single component dominates raw sum):
      w_alt       = 50   HARD BARRIER  (tanh-shaped cliff at ±50m)
      w_heading   = 30   Gaussian heading error
      w_pos       = 40   exp kernel to arc endpoint
      w_approach  = 15   velocity toward target
      w_nz        = 20   load factor band 3–7g
      w_bank      = 20   steep bank bonus φ>65°
      w_speed     = 10   speed band [230,290]
      w_time      = 15   time urgency (linear→exp after t_opt)
    """
    he_signed = herr(sv['psi'], spec.psi1)
    he_abs    = abs(he_signed)
    z_err     = sv['z'] - Z_CRUISE     # signed altitude deviation [m]
    z_err_abs = abs(z_err)
    t         = sv['elapsed']
    dist_init = spec.extra.get('dist_init', 1000.0)

    dist_now  = math.hypot(sv['x']-spec.x_target, sv['y']-spec.y_target)

    # ── 1. ALTITUDE  — tanh barrier  (FIX-3, hard wall) ─────────────────
    # Inside ±50m → positive reward peaking at 0 deviation
    # Outside →  steep negative cliff via tanh
    z_norm = z_err_abs / Z_BAND          # 0 at centre, 1 at wall, >1 outside
    if z_err_abs <= Z_BAND:
        R_alt = 1.0 - z_norm**2          # 0→1, peaks at centre
    else:
        # tanh cliff: drops steeply, then -1 -penalty*(excess)
        excess = z_err_abs - Z_BAND
        R_alt  = -1.0 - (excess / Z_BAND)**2   # goes to -2, -5, ...
    w_alt = 50.0

    # ── 2. HEADING  — Gaussian  ───────────────────────────────────────────
    sigma = max(5.0, spec.delta / 8.0)
    R_heading = math.exp(-0.5*(he_abs/sigma)**2)     # 0→1
    # Scale: 0 when error=3σ, 1 when on target
    w_heading = 30.0

    # ── 3. POSITION  — exponential kernel  ────────────────────────────────
    scale = max(dist_init*0.3, 100.0)
    R_pos = math.exp(-dist_now / scale)              # 0→1
    w_pos = 40.0 if curriculum_phase >= 1 else 0.0

    # ── 4. APPROACH VELOCITY  ─────────────────────────────────────────────
    psi_r    = math.radians(sv['psi'])
    vx       = sv['V'] * math.sin(psi_r)
    vy       = sv['V'] * math.cos(psi_r)
    dx_t     = spec.x_target - sv['x']
    dy_t     = spec.y_target - sv['y']
    d_horiz  = math.hypot(dx_t, dy_t) + 1e-6
    v_toward = (vx*dx_t + vy*dy_t) / d_horiz
    R_approach = np.clip(v_toward / V_CRUISE, -1., 1.)   # -1→+1
    w_approach = 15.0 if curriculum_phase >= 1 else 0.0

    # ── 5. LOAD FACTOR  — band reward [3–7g] ─────────────────────────────
    nz = sv['nz']
    if nz < NZ_MIN:                          # below floor → penalise
        R_nz = -1.0 * (NZ_MIN - nz) / NZ_MIN
    elif nz <= NZ_SWEET_LO:                  # 3→4.5g: ramp up
        R_nz = (nz - NZ_MIN) / (NZ_SWEET_LO - NZ_MIN)
    elif nz <= NZ_SWEET_HI:                  # 4.5→6.8g: full reward
        R_nz = 1.0
    elif nz <= NZ_MAX:                       # 6.8→7g: taper
        R_nz = 1.0 - (nz - NZ_SWEET_HI) / (NZ_MAX - NZ_SWEET_HI)
    else:                                    # >7g: hard cliff
        R_nz = -2.0 * (nz - NZ_MAX)
    w_nz = 20.0

    # ── 6. STEEP BANK BONUS  φ > 65° ─────────────────────────────────────
    phi_abs = abs(sv['phi'])
    if phi_abs >= PHI_AGGR:
        R_bank = (phi_abs - PHI_AGGR) / (PHI_LIMIT - PHI_AGGR)  # 0→1
    else:
        R_bank = -0.2 * (1.0 - phi_abs / PHI_AGGR)   # small nudge
    w_bank = 20.0 if curriculum_phase >= 1 else 5.0

    # ── 7. SPEED BAND  [230, 290] m/s ────────────────────────────────────
    v_dev  = max(0., abs(sv['V'] - V_CRUISE) - 30.)
    R_speed = np.clip(1.0 - (v_dev/30.)**2, -1., 1.)
    w_speed = 10.0

    # ── 8. TIME URGENCY ───────────────────────────────────────────────────
    if curriculum_phase >= 1:
        if t <= spec.t_opt:
            R_time = 1.0 - t/spec.t_opt              # 1→0 before t_opt
        else:
            R_time = -((t - spec.t_opt) / spec.t_opt)**2  # quadratic after
    else:
        R_time = 0.0
    w_time = 15.0

    # ── 9. TERMINAL ───────────────────────────────────────────────────────
    R_terminal = 0.0
    if cause == 'success':
        time_bonus = max(0., 500.*(1. - t/spec.t_opt)) if t<=spec.t_opt else 100.
        pos_bonus  = max(0., 300.*(1. - dist_now/max(dist_init*0.05, 50.)))
        alt_bonus  = max(0., 100.*(1. - z_err_abs/Z_BAND))
        R_terminal = 1000. + time_bonus + pos_bonus + alt_bonus
    elif cause == 'alt_violation':
        R_terminal = -600.                   # FIX-4 severe
    elif cause == 'timeout':
        partial    = max(0., 1. - he_abs/spec.delta)
        R_terminal = -100. + 80.*partial
    elif cause == 'crash':
        R_terminal = -800.

    step_reward = (w_alt*R_alt + w_heading*R_heading + w_pos*R_pos
                   + w_approach*R_approach + w_nz*R_nz + w_bank*R_bank
                   + w_speed*R_speed + w_time*R_time + R_terminal)

    info = dict(alt=R_alt*w_alt, heading=R_heading*w_heading,
                pos=R_pos*w_pos, approach=R_approach*w_approach,
                nz=R_nz*w_nz, bank=R_bank*w_bank,
                speed=R_speed*w_speed, time=R_time*w_time,
                terminal=R_terminal)
    return float(step_reward), info


# ══════════════════════════════════════════════════════════════════════════════
# ENVIRONMENT  v3
# ══════════════════════════════════════════════════════════════════════════════

class HeadingChangeEnv:
    def __init__(self, spec: ManeuverSpec, curriculum_phase=1):
        self.spec = spec
        self.dyn  = F18_6DOF()
        self.curriculum_phase = curriculum_phase
        self._sv = self._sv_prev = None
        self._suc_ctr = self._alt_ctr = 0

    def reset(self):
        self.dyn.reset(psi0=self.spec.psi0)
        self._sv = self.dyn.sv()
        self._sv_prev = self._sv.copy()
        self._suc_ctr = self._alt_ctr = 0
        return self.dyn.state_vec()

    def step(self, action):
        a = np.clip(action, -1., 1.)
        # FIX-1: map normalised action to physical commands
        psi_dot_cmd  = float(a[0]) * A_PSI_DOT     # deg/s
        bank_rate_cmd= float(a[1]) * A_BANK_RATE    # deg/s
        hdot_cmd     = float(a[2]) * A_DZ_TRIM      # m/s

        self._sv_prev = self._sv.copy()
        self._sv = self.dyn.step(psi_dot_cmd, bank_rate_cmd, hdot_cmd)

        he_abs  = abs(herr(self._sv['psi'], self.spec.psi1))
        z_err   = abs(self._sv['z'] - Z_CRUISE)
        t       = self._sv['elapsed']

        # Altitude strike counter  (FIX-4: alt_strikes=3)
        if z_err > Z_BAND:
            self._alt_ctr += 1
        else:
            self._alt_ctr = max(0, self._alt_ctr - 1)

        done=False; cause='running'
        if self._sv['z'] < 3000. or self._sv['V'] < F18_6DOF.V_MIN:
            done, cause = True, 'crash'
        elif self._alt_ctr >= self.spec.alt_strikes:
            done, cause = True, 'alt_violation'
        elif t >= self.spec.t_limit:
            done, cause = True, 'timeout'
        else:
            if he_abs <= self.spec.success_herr and z_err <= Z_BAND:
                self._suc_ctr += 1
                if self._suc_ctr >= self.spec.success_hold:
                    done, cause = True, 'success'
            else:
                self._suc_ctr = 0

        reward, rew_info = compute_reward(
            self._sv_prev, self._sv, self.spec, a, done, cause,
            self.curriculum_phase)

        obs = self.dyn.state_vec()
        info = dict(cause=cause, herr=he_abs, z_err=z_err,
                    z=self._sv['z'], V=self._sv['V'],
                    phi=self._sv['phi'], nz=self._sv['nz'],
                    t=t, x=self._sv['x'], y=self._sv['y'],
                    psi=self._sv['psi'], rew_info=rew_info)
        return obs, reward, done, info

    @property
    def state(self): return self._sv.copy()


# ══════════════════════════════════════════════════════════════════════════════
# DDPG NETWORKS  (wider: 512-512 for better capacity)
# ══════════════════════════════════════════════════════════════════════════════

def mlp(in_d, out_d, hidden=(512,512)):
    layers, prev = [], in_d
    for h in hidden:
        layers += [nn.Linear(prev,h), nn.LayerNorm(h), nn.ReLU()]
        prev = h
    layers.append(nn.Linear(prev,out_d))
    return nn.Sequential(*layers)


class Actor(nn.Module):
    def __init__(self, s=STATE_DIM, a=ACTION_DIM):
        super().__init__()
        self.net = mlp(s, a)
        nn.init.uniform_(self.net[-1].weight, -3e-3, 3e-3)
        nn.init.uniform_(self.net[-1].bias,   -3e-3, 3e-3)
    def forward(self, s): return torch.tanh(self.net(s))
    @torch.no_grad()
    def act(self, obs, noise=0., det=False):
        s = torch.FloatTensor(obs).unsqueeze(0).to(DEVICE)
        a = self.forward(s).squeeze(0).cpu().numpy()
        if not det and noise>0:
            a += np.random.normal(0., noise, a.shape)
        return a.clip(-1.,1.).astype(np.float32)


class Critic(nn.Module):
    def __init__(self, s=STATE_DIM, a=ACTION_DIM):
        super().__init__()
        self.net = mlp(s+a, 1)
    def forward(self, s, a): return self.net(torch.cat([s,a],-1))


class ReplayBuffer:
    def __init__(self, s=STATE_DIM, a=ACTION_DIM, cap=300_000):
        self.cap=cap; self.ptr=self.size=0
        self.S =np.zeros((cap,s),np.float32); self.A =np.zeros((cap,a),np.float32)
        self.R =np.zeros((cap,1),np.float32); self.S2=np.zeros((cap,s),np.float32)
        self.D =np.zeros((cap,1),np.float32)
    def add(self,s,a,r,s2,d):
        i=self.ptr%self.cap
        self.S[i]=s;self.A[i]=a;self.R[i]=r;self.S2[i]=s2;self.D[i]=d
        self.ptr=(self.ptr+1)%self.cap; self.size=min(self.size+1,self.cap)
    def sample(self,n):
        idx=np.random.randint(0,self.size,n)
        t=lambda x: torch.FloatTensor(x[idx]).to(DEVICE)
        return t(self.S),t(self.A),t(self.R),t(self.S2),t(self.D)


class DDPGAgent:
    def __init__(self, alr=3e-4, clr=1e-3, gamma=0.99, tau=0.005, batch=256):
        self.gamma=gamma; self.tau=tau; self.batch=batch
        self.actor   = Actor().to(DEVICE)
        self.actor_t = copy.deepcopy(self.actor)
        self.critic  = Critic().to(DEVICE)
        self.critic_t= copy.deepcopy(self.critic)
        self.opt_a   = optim.Adam(self.actor.parameters(),  lr=alr)
        self.opt_c   = optim.Adam(self.critic.parameters(), lr=clr)
        self.buf     = ReplayBuffer()
        self.c_losses=[]; self.a_losses=[]

    def act(self, obs, noise=0., det=False): return self.actor.act(obs,noise,det)

    def _soft(self,n,t):
        for p,tp in zip(n.parameters(),t.parameters()):
            tp.data.copy_(self.tau*p.data+(1-self.tau)*tp.data)

    def update(self):
        if self.buf.size < self.batch: return
        S,A,R,S2,D = self.buf.sample(self.batch)
        with torch.no_grad():
            Qt = R + self.gamma*(1-D)*self.critic_t(S2, self.actor_t(S2))
        lc = F.mse_loss(self.critic(S,A), Qt)
        self.opt_c.zero_grad(); lc.backward(); self.opt_c.step()
        la = -self.critic(S, self.actor(S)).mean()
        self.opt_a.zero_grad(); la.backward(); self.opt_a.step()
        self._soft(self.actor,self.actor_t); self._soft(self.critic,self.critic_t)
        self.c_losses.append(float(lc)); self.a_losses.append(float(la))

    def save(self,p): torch.save({'a':self.actor.state_dict(),'c':self.critic.state_dict()},p)
    def load(self,p):
        ck=torch.load(p,map_location=DEVICE)
        self.actor.load_state_dict(ck['a']); self.actor_t=copy.deepcopy(self.actor)
        self.critic.load_state_dict(ck['c']); self.critic_t=copy.deepcopy(self.critic)


# ══════════════════════════════════════════════════════════════════════════════
# TRAINING  with curriculum  (FIX-7)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class TrainCfg:
    n_episodes      : int   = 6000
    warmup_steps    : int   = 3000
    noise_start     : float = 0.40
    noise_end       : float = 0.04
    updates_per_step: int   = 2       # more updates per env step
    print_every     : int   = 500
    eval_every      : int   = 1000
    n_eval          : int   = 30
    curriculum_switch: int  = 1000    # FIX-7
    save_dir        : str   = 'ckpt_v3'


def _eval(spec, agent, n=30, phase=1):
    env = HeadingChangeEnv(spec, curriculum_phase=phase)
    rs,ss,hs,zs,ts,ps=[],[],[],[],[],[]
    for _ in range(n):
        obs=env.reset(); done=False; ep_r=0.
        while not done:
            a=agent.act(obs,det=True); obs,r,done,info=env.step(a); ep_r+=r
        rs.append(ep_r); ss.append(int(info['cause']=='success'))
        hs.append(info['herr']); zs.append(info['z_err'])
        ts.append(info['t']); ps.append(abs(info['phi']))
    return np.mean(rs),np.mean(ss),np.mean(hs),np.mean(zs),np.mean(ts),np.mean(ps)


def train_one(spec: ManeuverSpec, cfg: TrainCfg):
    agent = DDPGAgent()
    os.makedirs(cfg.save_dir, exist_ok=True)

    stats = dict(reward=[],herr=[],z_err=[],success=[],cause=[],
                 alt_viol=[],turn_t=[],phi_peak=[],nz_peak=[])
    best=-np.inf; total=0

    print(f"\n{'═'*65}")
    print(f"  {spec.key}  {spec.label}  [{spec.category.upper()}]")
    print(f"  T_opt={spec.t_opt:.1f}s  T_lim={spec.t_limit:.1f}s  "
          f"phi_opt={spec.phi_opt:.0f}°  nz_design={spec.nz_design:.2f}g")
    print(f"  arc_target=({spec.x_target:.0f},{spec.y_target:.0f})m  "
          f"dist_init={spec.extra['dist_init']:.0f}m")
    print(f"  Curriculum: phase 0 ep<{cfg.curriculum_switch} → "
          f"phase 1 ep≥{cfg.curriculum_switch}")
    print(f"{'═'*65}")
    t0=time.time()

    for ep in range(1, cfg.n_episodes+1):
        phase = 0 if ep < cfg.curriculum_switch else 1
        env   = HeadingChangeEnv(spec, curriculum_phase=phase)
        obs   = env.reset(); done=False; ep_r=0.
        frac  = min(ep/cfg.n_episodes,1.)
        noise = cfg.noise_start + frac*(cfg.noise_end-cfg.noise_start)
        ep_z=[]; ep_phi=[]; ep_nz=[]

        while not done:
            if total < cfg.warmup_steps:
                # FIX-1: smart warmup — bias toward high bank angles
                a = np.array([
                    np.random.uniform(0.5,1.0),   # positive turn rate
                    np.random.uniform(-0.3,0.3),  # small bank rate variation
                    np.random.uniform(-0.2,0.2),  # minimal altitude trim
                ], dtype=np.float32)
            else:
                a = agent.act(obs, noise)
            obs2,r,done,info = env.step(a)
            agent.buf.add(obs,a,r,obs2,float(done))
            obs=obs2; ep_r+=r; total+=1
            ep_z.append(info['z_err']); ep_phi.append(abs(info['phi']))
            ep_nz.append(info['nz'])
            if total >= cfg.warmup_steps:
                for _ in range(cfg.updates_per_step):
                    agent.update()

        stats['reward'].append(ep_r)
        stats['herr'].append(info['herr'])
        stats['z_err'].append(np.mean(ep_z))
        stats['success'].append(int(info['cause']=='success'))
        stats['cause'].append(info['cause'])
        stats['alt_viol'].append(int(info['cause']=='alt_violation'))
        stats['turn_t'].append(info['t'])
        stats['phi_peak'].append(np.max(ep_phi) if ep_phi else 0.)
        stats['nz_peak'].append(np.mean(ep_nz) if ep_nz else 0.)

        if ep % cfg.print_every == 0:
            w=cfg.print_every
            ar =np.mean(stats['reward'][-w:])
            sr =100*np.mean(stats['success'][-w:])
            av =100*np.mean(stats['alt_viol'][-w:])
            he =np.mean(stats['herr'][-w:])
            ze =np.mean(stats['z_err'][-w:])
            ph =np.mean(stats['phi_peak'][-w:])
            nzm=np.mean(stats['nz_peak'][-w:])
            tt =np.mean(stats['turn_t'][-w:])
            flag='✓' if sr>0 else '✗'
            print(f"  ep={ep:>5} [{phase}] R={ar:>8.1f}  suc={sr:>5.1f}%{flag} "
                  f"altV={av:>5.1f}%  herr={he:>5.1f}°  "
                  f"z={ze:>5.1f}m  φ={ph:>5.1f}°  nz={nzm:>4.1f}g  "
                  f"t={tt:>5.1f}s  σ={noise:.3f}  [{time.time()-t0:.0f}s]")

        if ep % cfg.eval_every == 0:
            er,es,eh,ez,et,ep_=_eval(spec,agent,cfg.n_eval,phase=1)
            sub5 = '✓<5s' if et<=5.0 else f'✗{et:.1f}s'
            print(f"  [EVAL] R={er:.0f} suc={es*100:.0f}% herr={eh:.1f}° "
                  f"z_err={ez:.1f}m t={et:.1f}s φ={ep_:.0f}° {sub5}")
            if er > best:
                best=er; agent.save(os.path.join(cfg.save_dir,f'{spec.key}.pt'))
                print(f"  [SAVE] new best R={best:.0f}")

    return agent, stats


def train_all(keys=None, cfg=None):
    cfg  = cfg or TrainCfg()
    keys = keys or list(ALL_SPECS.keys())
    print("\n" + "╔"+"═"*63+"╗")
    print("║  DDPG v3 — FIXED ACTION SPACE + CURRICULUM + BARRIERS       ║")
    print("║  Actions: (psi_dot_cmd, bank_rate_cmd, hdot_trim)            ║")
    print("║  Altitude: proportional hold + agent trim (max ±3 m/s)       ║")
    print("║  Phase 0: altitude+heading only  →  Phase 1: full reward      ║")
    print("╚"+"═"*63+"╝")
    results={}
    for k in keys: results[k]=train_one(ALL_SPECS[k], cfg)
    return results


# ══════════════════════════════════════════════════════════════════════════════
# RECORD & PLOT
# ══════════════════════════════════════════════════════════════════════════════

def record(spec, agent):
    env=HeadingChangeEnv(spec, curriculum_phase=1)
    obs=env.reset(); done=False; sv0=env.state
    traj={k:[sv0[k]] for k in sv0}
    traj.update(time=[0.],herr=[abs(herr(sv0['psi'],spec.psi1))],
                z_err=[0.],reward=[],rew_alt=[],rew_heading=[],
                rew_pos=[],rew_time=[],rew_bank=[],rew_nz=[])
    while not done:
        a=agent.act(obs,det=True); obs,r,done,info=env.step(a)
        sv=env.state
        for k in sv: traj[k].append(sv[k])
        traj['time'].append(sv['elapsed'])
        traj['herr'].append(info['herr'])
        traj['z_err'].append(info['z_err'])
        traj['reward'].append(r)
        ri=info['rew_info']
        for c in ('alt','heading','pos','time','bank','nz'):
            traj[f'rew_{c}'].append(ri.get(c,0.))
    traj['termination']=info['cause']; traj['spec']=spec
    return traj


plt.rcParams.update({
    'figure.facecolor':'#0d1117','axes.facecolor':'#161b22',
    'axes.edgecolor':'#30363d','axes.labelcolor':'#c9d1d9',
    'xtick.color':'#8b949e','ytick.color':'#8b949e','text.color':'#c9d1d9',
    'grid.color':'#21262d','grid.linewidth':0.6,
    'legend.facecolor':'#161b22','legend.edgecolor':'#30363d',
    'font.family':'monospace',
})
CAT_COL={'nav':'#00d4ff','aggressive':'#fd9644','extreme':'#ff453a'}


def plot_trajectory(traj, save=True):
    spec=traj['spec']; t=np.array(traj['time']); col=CAT_COL[spec.category]
    z_arr=np.array(traj['z']); nz_arr=np.array(traj['nz'])
    sub5=(traj['termination']=='success' and t[-1]<=5.0)
    alt_ok=(np.max(np.abs(z_arr-Z_CRUISE))<=Z_BAND)
    nz_ok=(np.max(nz_arr)<=NZ_MAX and np.min(nz_arr)>=NZ_MIN)

    fig=plt.figure(figsize=(24,15)); fig.patch.set_facecolor('#0d1117')
    status_parts=[traj['termination'].upper()]
    if sub5:     status_parts.append('✓ SUB-5s')
    if alt_ok:   status_parts.append('✓ ALT')
    if nz_ok:    status_parts.append('✓ NZ')
    fig.suptitle(f"{spec.label} [{spec.category.upper()}]  |  "
                 f"{' | '.join(status_parts)}  |  "
                 f"t={t[-1]:.1f}s  herr={traj['herr'][-1]:.1f}°  "
                 f"z_max_err={np.max(np.abs(z_arr-Z_CRUISE)):.1f}m  "
                 f"nz_max={np.max(nz_arr):.2f}g",
                 fontsize=10,fontweight='bold',color='#e6edf3',y=1.0)

    gs=gridspec.GridSpec(3,4,figure=fig,hspace=0.52,wspace=0.38)
    def ax(r,c,cs=1): return fig.add_subplot(gs[r,c:c+cs])

    # Heading
    a=ax(0,0,2)
    a.plot(t,traj['psi'],color=col,lw=2.2,label='ψ [°]')
    a.axhline(spec.psi1,color='#32d74b',lw=1.5,ls='--',label=f'Target {spec.psi1:.0f}°')
    a.axhline(spec.psi0,color='white',lw=0.8,ls=':',alpha=0.4)
    a.axvline(spec.t_opt,color='#ffd60a',lw=1.,ls=':',label=f'T_opt={spec.t_opt:.1f}s')
    a.axvline(5.0,color='#ff453a',lw=1.5,ls='--',label='5s deadline')
    a.set_title("Heading ψ [°]  →  must reach target"); a.grid(True); a.legend(fontsize=7)

    # Altitude — with constraint band prominently shown
    a=ax(0,2,2)
    a.plot(t,z_arr,color='#7cfc00',lw=2.5,label='z [m]')
    a.axhline(Z_CRUISE,color='white',lw=1.,ls='--',alpha=0.6,label='5000m cruise')
    a.axhline(Z_CRUISE+Z_BAND,color='#ff453a',lw=2.,ls='--',label='±50m HARD WALL')
    a.axhline(Z_CRUISE-Z_BAND,color='#ff453a',lw=2.,ls='--')
    a.fill_between(t,Z_CRUISE-Z_BAND,Z_CRUISE+Z_BAND,alpha=0.10,color='#32d74b')
    bad=np.abs(z_arr-Z_CRUISE)>Z_BAND
    if bad.any():
        a.fill_between(t,z_arr,Z_CRUISE,where=bad,alpha=0.4,color='#ff453a',label='VIOLATION')
    a.set_ylim(Z_CRUISE-200,Z_CRUISE+200)
    a.set_title("Altitude z [m]  ← HARD wall ±50m  (FIX-2: autopilot hold)")
    a.grid(True); a.legend(fontsize=7)

    # Bank angle
    a=ax(1,0)
    a.plot(t,traj['phi'],color='#bf5af2',lw=2.)
    a.axhspan(PHI_AGGR,PHI_LIMIT,alpha=0.12,color='#fd9644',label=f'>={PHI_AGGR:.0f}° bonus')
    a.axhspan(-PHI_LIMIT,-PHI_AGGR,alpha=0.12,color='#fd9644')
    a.axhline(PHI_OPT_DEG,color='white',lw=0.8,ls=':',alpha=0.6,label=f'φ_opt={PHI_OPT_DEG:.0f}°')
    a.set_title("Bank φ [°]  (orange zone = aggression bonus)"); a.grid(True); a.legend(fontsize=7)

    # Load factor
    a=ax(1,1)
    a.plot(t,nz_arr,color='#30d158',lw=2.)
    a.axhspan(NZ_SWEET_LO,NZ_SWEET_HI,alpha=0.18,color='#30d158',label=f'{NZ_SWEET_LO}–{NZ_SWEET_HI}g sweet')
    a.axhspan(NZ_MIN,NZ_SWEET_LO,alpha=0.08,color='#ffd60a',label=f'{NZ_MIN}–{NZ_SWEET_LO}g ramp')
    a.axhline(NZ_MAX,color='#ff453a',lw=2.,ls='--',label=f'{NZ_MAX}g LIMIT')
    a.axhline(NZ_MIN,color='#ffd60a',lw=1.2,ls='--',label=f'{NZ_MIN}g floor')
    a.set_ylim(0,9); a.set_title("Load factor n [g]  green=sweet 4.5–6.8g")
    a.grid(True); a.legend(fontsize=7)

    # Speed
    a=ax(1,2)
    a.plot(t,traj['V'],color='#ff6b35',lw=2.)
    a.axhline(V_CRUISE,color='white',lw=0.8,ls='--',alpha=0.5)
    a.axhline(V_CRUISE-30,color='#ff453a',lw=1.2,ls='--',label='±30m/s band')
    a.axhline(V_CRUISE+30,color='#ff453a',lw=1.2,ls='--')
    a.set_title("TAS V [m/s]"); a.grid(True); a.legend(fontsize=7)

    # Reward components
    a=ax(1,3)
    t_a=np.array(traj['time'][1:])
    RMAP={'alt':'#ff375f','heading':'#ffd60a','pos':'#64d2ff',
          'time':'#30d158','bank':'#fd9644','nz':'#bf5af2'}
    for c,cc in RMAP.items():
        arr=traj.get(f'rew_{c}',[])
        if arr: a.plot(t_a,arr,color=cc,lw=1.4,label=c)
    a.axhline(0,color='white',lw=0.4,alpha=0.3)
    a.axvline(5.,color='#ff453a',lw=1.,ls='--',alpha=0.7,label='5s')
    a.set_title("Reward components"); a.grid(True); a.legend(fontsize=6)

    # Heading error decay
    a=ax(2,0)
    a.fill_between(t,traj['herr'],alpha=0.2,color=col)
    a.plot(t,traj['herr'],color=col,lw=2.)
    a.axhline(spec.success_herr,color='#32d74b',lw=1.5,ls='--',label=f'{spec.success_herr:.0f}° gate')
    a.axvline(5.0,color='#ff453a',lw=1.5,ls='--',label='5s')
    a.set_title("Heading error [°]  must reach green line"); a.grid(True); a.legend(fontsize=7)

    # Alt error
    a=ax(2,1)
    ze=np.array(traj['z_err'])
    a.fill_between(t,ze,alpha=0.25,color='#7cfc00')
    a.plot(t,ze,color='#7cfc00',lw=2.)
    a.axhline(Z_BAND,color='#ff453a',lw=2.,ls='--',label='50m WALL')
    a.set_ylim(0,max(Z_BAND*3,ze.max()*1.1))
    a.set_title("Altitude error |z-5000| [m]  must stay below red"); a.grid(True); a.legend(fontsize=7)

    # Ground track
    a=ax(2,2,2)
    x_km=np.array(traj['x'])/1000; y_km=np.array(traj['y'])/1000
    sc=a.scatter(x_km,y_km,c=t,cmap='plasma',s=15,linewidths=0,zorder=3)
    a.plot(x_km,y_km,color='white',lw=0.5,alpha=0.3)
    a.plot(x_km[0],y_km[0],'o',color='#32d74b',ms=10,label='start',zorder=5)
    a.plot(x_km[-1],y_km[-1],'s',color='#ff453a',ms=10,label='end',zorder=5)
    a.plot(spec.x_target/1000,spec.y_target/1000,'*',color='#ffd60a',ms=16,
           label='arc target ★',zorder=5)
    # Draw ideal arc
    arc_t=np.linspace(0,spec.delta,60)
    arc_x=[]; arc_y=[]
    for ang in arc_t:
        xe,yn=_arc_endpoint(spec.psi0,ang,spec.R)
        arc_x.append(xe/1000); arc_y.append(yn/1000)
    a.plot(arc_x,arc_y,color='#32d74b',lw=1.5,ls=':',alpha=0.6,label='ideal arc')
    plt.colorbar(sc,ax=a,label='t [s]')
    a.set_title("Ground track [km]  dotted=ideal arc"); a.grid(True)
    a.set_aspect('equal','datalim'); a.legend(fontsize=7)
    a.set_xlabel("E [km]"); a.set_ylabel("N [km]")

    if save:
        fname=f"traj_{spec.key}_v3.png"
        plt.savefig(fname,dpi=130,bbox_inches='tight',facecolor=fig.get_facecolor())
        plt.close(); print(f"[Saved] {fname}")
    else:
        plt.show()


def plot_training(all_results):
    n=len(all_results)
    cols=min(n,5); rows=(n+cols-1)//cols
    fig,axes=plt.subplots(rows,cols,figsize=(5*cols,4*rows))
    fig.patch.set_facecolor('#0d1117')
    fig.suptitle("DDPG v3 Training — Success Rate & Altitude Violations",
                 fontsize=13,fontweight='bold',color='#c9d1d9')
    axes_flat=np.array(axes).flat
    for k,(agent,stats) in all_results.items():
        ax=next(axes_flat); spec=ALL_SPECS[k]
        col=CAT_COL[spec.category]
        suc=np.array(stats['success'],float); n_ep=len(suc)
        av=np.array(stats['alt_viol'],float)
        w=min(300,max(1,n_ep//10))
        sm_s=np.convolve(suc,np.ones(w)/w,'valid')*100
        sm_a=np.convolve(av, np.ones(w)/w,'valid')*100
        xs=np.linspace(0,n_ep,len(sm_s))
        ax.plot(xs,sm_s,color=col,lw=2.,label='success%')
        ax.plot(xs,sm_a,color='#ff453a',lw=1.5,ls='--',label='altViol%')
        ax.set_ylim(0,105); ax.grid(True)
        ax.set_title(f"{spec.psi0}°→{spec.psi1}°",color=col,fontsize=9)
        ax.legend(fontsize=6)
        fin=100*np.mean(stats['success'][-200:])
        fin_av=100*np.mean(stats['alt_viol'][-200:])
        ax.text(0.97,0.90,f"suc={fin:.0f}%",transform=ax.transAxes,
                ha='right',fontsize=8,color=col)
        ax.text(0.97,0.78,f"altV={fin_av:.0f}%",transform=ax.transAxes,
                ha='right',fontsize=7,color='#ff453a')
    for ax in axes_flat: ax.set_visible(False)
    plt.tight_layout()
    plt.savefig("ddpg_v3_training.png",dpi=130,bbox_inches='tight',
                facecolor=fig.get_facecolor())
    plt.close(); print("[Saved] ddpg_v3_training.png")


def constraint_report(traj):
    spec=traj['spec']; t=np.array(traj['time'])
    z=np.array(traj['z']); nz=np.array(traj['nz'])
    z_err=np.abs(z-Z_CRUISE); V_arr=np.array(traj['V'])
    psi=np.array(traj['psi']); phi=np.array(traj['phi'])

    t_hit=None
    for i,p in enumerate(psi):
        if abs(p-spec.psi1)<spec.success_herr and i>0: t_hit=t[i]; break

    print(f"\n  ┌{'─'*58}┐")
    print(f"  │  CONSTRAINT REPORT v3: {spec.label:<35}│")
    print(f"  ├{'─'*58}┤")
    #s1=f'{t_hit:.2f}s  {"✓ SUB-5s" if t_hit and t_hit<=5. else "✗ LATE" if t_hit else "not reached"}'
    print(f"  │  C0  <5s deadline    : {s1:<34}│")
    s2=f'{z_err.max():.1f}m  {"✓ OK" if z_err.max()<=Z_BAND else "*** VIOLATED ***"}'
    print(f"  │  C1  alt max error   : {s2:<34}│")
    s3=f'{z_err.mean():.1f}m'
    print(f"  │  C1  alt mean error  : {s3:<34}│")
    s4=f'{nz.max():.2f}g  {"✓ OK" if nz.max()<=NZ_MAX else "*** EXCEEDED ***"}'
    print(f"  │  C2  nz max          : {s4:<34}│")
    s5=f'{nz.min():.2f}g  {"✓ OK" if nz.min()>=NZ_MIN else "*** BELOW FLOOR ***"}'
    print(f"  │  C2  nz min          : {s5:<34}│")
    sweet_pct=100*np.mean((nz>=NZ_SWEET_LO)&(nz<=NZ_SWEET_HI))
    print(f"  │  C2  nz sweet %      : {sweet_pct:.1f}% in {NZ_SWEET_LO}–{NZ_SWEET_HI}g band{'':<9}│")
    vok=V_arr.min()>=V_CRUISE-30 and V_arr.max()<=V_CRUISE+30
    s6=f'[{V_arr.min():.0f},{V_arr.max():.0f}] m/s  {"✓ OK" if vok else "✗ CHECK"}'
    print(f"  │  C3  speed band      : {s6:<34}│")
    dist_f=math.hypot(traj['x'][-1]-spec.x_target, traj['y'][-1]-spec.y_target)
    s7=f'{dist_f:.0f}m from target  {"✓ OK" if dist_f<300 else "✗ FAR"}'
    print(f"  │  C4  arc-pt distance : {s7:<34}│")
    phi_max=np.max(np.abs(phi))
    s8=f'{phi_max:.1f}°  {"✓ AGGRESSIVE" if phi_max>=PHI_AGGR else "✗ shallow"}'
    print(f"  │  C5  bank max        : {s8:<34}│")
    print(f"  │  C6  termination     : {traj['termination'].upper():<34}│")
    print(f"  └{'─'*58}┘")


def print_table(all_results):
    print("\n"+"═"*90)
    print(f"  {'Key':<12}{'Label':<22}{'Cat':<11}"
          f"{'T_opt':>6}{'Suc%':>6}{'HErr°':>7}{'AltErr':>7}{'t[s]':>7}{'nz':>5}{'φmax':>6}{'<5?':>5}")
    print("═"*90)
    for k,(agent,stats) in all_results.items():
        spec=ALL_SPECS[k]; w=min(300,len(stats['success']))
        sr=100*np.mean(stats['success'][-w:])
        he=np.mean(stats['herr'][-w:])
        ze=np.mean(stats['z_err'][-w:])
        tt=np.mean(stats['turn_t'][-w:])
        ph=np.mean(stats['phi_peak'][-w:])
        nzm=np.mean(stats['nz_peak'][-w:])
        print(f"  {k:<12}{spec.label:<22}{spec.category:<11}"
              f"{spec.t_opt:>6.1f}{sr:>6.1f}{he:>7.1f}{ze:>7.1f}"
              f"{tt:>7.1f}{nzm:>5.1f}{ph:>6.1f}{'✓' if tt<=5. else '✗':>5}")
    print("═"*90)


# ══════════════════════════════════════════════════════════════════════════════
# MAIN
# ══════════════════════════════════════════════════════════════════════════════

if __name__ == "__main__":
    torch.manual_seed(42); np.random.seed(42); random.seed(42)

    TRAIN_KEYS = ['H000_030']

    cfg = TrainCfg(
        n_episodes       = 6000,
        warmup_steps     = 3000,
        noise_start      = 0.40,
        noise_end        = 0.04,
        updates_per_step = 2,
        print_every      = 500,
        eval_every       = 1000,
        n_eval           = 30,
        curriculum_switch= 1000,
        save_dir         = 'ckpt_v3',
    )

    all_results = train_all(TRAIN_KEYS, cfg)
    print_table(all_results)
    plot_training(all_results)

    trajectories={}
    for k,(agent,_) in all_results.items():
        traj=record(ALL_SPECS[k], agent)
        trajectories[k]=traj
        constraint_report(traj)
        plot_trajectory(traj, save=True)

    print("\n✓ DDPG v3 complete.")

[DDPG-v3]  PyTorch 2.10.0+cpu  device=cpu
[Physics] PHI_OPT=80.0°  PSI_DOT_MAX=12.3°/s  R_OPT=1215m  NZ_design=5.76g
[Actions] psi_dot=±12.3°/s  bank_rate=±60°/s  dz_trim=±3.0m/s
[Specs] 21 maneuvers built

╔═══════════════════════════════════════════════════════════════╗
║  DDPG v3 — FIXED ACTION SPACE + CURRICULUM + BARRIERS       ║
║  Actions: (psi_dot_cmd, bank_rate_cmd, hdot_trim)            ║
║  Altitude: proportional hold + agent trim (max ±3 m/s)       ║
║  Phase 0: altitude+heading only  →  Phase 1: full reward      ║
╚═══════════════════════════════════════════════════════════════╝

═════════════════════════════════════════════════════════════════
  H000_030  0°→30° (30°)  [NAV]
  T_opt=2.4s  T_lim=15.0s  phi_opt=80°  nz_design=5.76g
  arc_target=(163,608)m  dist_init=629m
  Curriculum: phase 0 ep<1000 → phase 1 ep≥1000
═════════════════════════════════════════════════════════════════
  ep=  500 [0] R=  -118.6  suc=  0.0%✗ altV= 65.0%  herr= 30.2°  z= 31.3m  φ= 80.0°  nz= 1.8

NameError: name 's1' is not defined